# CORDIS Collaboration Network — Example Solution

This notebook is the **reference solution** to the *Your Turn* exercise in `networkx.ipynb`.
It builds a research organisation collaboration network from the CORDIS EU funding dataset,
filtered to `masterCall == 'HORIZON-CL5-2022-D2-01'` (Clean Energy, Horizon Europe 2022).

## Contents
1. Load & explore the data  
2. Prepare: filter, extract country, compute per-org totals  
3. Build the collaboration edge list  
4. Basic network graph  
5. Edge width by number of collaborations  
6. Node size by EC funding  
7. Node colour by country  
8. Fully styled network (all three encodings)  
9. Community detection  
10. Centrality analysis  

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from itertools import combinations
from collections import Counter

print(f"NetworkX: {nx.__version__}  |  Pandas: {pd.__version__}")

---
## 1. Load & Explore the Data

Each row in `cordis_orgs.csv` is one organisation's participation in one EU project.

| Column | Description |
|---|---|
| `projectID` | Unique project identifier |
| `organisationID` | Unique organisation identifier |
| `shortName` | Short label (e.g. *INRIA*, *CNRS*) |
| `ecContribution` | EC funding awarded to this org for this project (€) |
| `vatNumber` | VAT number — first 2 letters encode the country (e.g. `FR`, `DE`) |
| `masterCall` | The top-level Horizon Europe call identifier |

In [ ]:
df_all = pd.read_csv('cordis_orgs.csv')

print(f"Full dataset — rows: {len(df_all):,}  projects: {df_all['projectID'].nunique():,}  orgs: {df_all['organisationID'].nunique():,}")
print(f"\nTop master calls:")
print(df_all['masterCall'].value_counts().head(10).to_string())
df_all.head(3)

---
## 2. Prepare: Filter, Extract Country, Compute Per-Org Totals

In [ ]:
# Filter to a single master call
MASTER_CALL = 'HORIZON-CL5-2022-D2-01'
df = df_all[df_all['masterCall'] == MASTER_CALL].copy()
print(f"Filtered to '{MASTER_CALL}'")
print(f"Rows: {len(df):,}  |  Projects: {df['projectID'].nunique():,}  |  Orgs: {df['organisationID'].nunique():,}")

# Extract 2-letter country code from VAT number prefix
df['country'] = df['vatNumber'].str.extract(r'^([A-Z]{2})')

# Use shortName where available; fall back to first 25 chars of full name
df['label'] = df['shortName'].where(
    df['shortName'].notna() & (df['shortName'].str.strip() != ''),
    df['name'].str[:25]
)

# Per-org node attributes
org_attrs = (df.groupby('organisationID')
               .agg(label=('label', 'first'),
                    country=('country', 'first'),
                    total_ec=('ecContribution', 'sum'))
               .reset_index())

print(f"\nSample org attributes:")
print(org_attrs.head(8).to_string(index=False))

---
## 3. Build the Collaboration Edge List

For each project, create a link between every pair of participating organisations.
Count how many projects each pair shares → stored as the `collaborations` edge weight.

In [ ]:
edge_records = []
for pid, group in df.groupby('projectID'):
    org_ids = group['organisationID'].tolist()
    for a, b in combinations(org_ids, 2):
        if a > b:
            a, b = b, a  # canonical ordering
        edge_records.append((a, b))

edge_df = (pd.DataFrame(edge_records, columns=['org_a', 'org_b'])
             .groupby(['org_a', 'org_b'])
             .size()
             .reset_index(name='collaborations'))

print(f"Unique collaboration pairs: {len(edge_df):,}")
print(f"Max collaborations between a pair: {edge_df['collaborations'].max()}")

# Top 10 most frequent pairs
label_map = org_attrs.set_index('organisationID')['label'].to_dict()
top_pairs = edge_df.nlargest(10, 'collaborations').copy()
top_pairs['org_a_name'] = top_pairs['org_a'].map(label_map)
top_pairs['org_b_name'] = top_pairs['org_b'].map(label_map)
print("\nTop 10 most frequent collaborating pairs:")
print(top_pairs[['org_a_name', 'org_b_name', 'collaborations']].to_string(index=False))

---
## 4. Basic Network Graph

Build the full graph, then work with the **top 60 orgs by degree** for readable layout.

In [ ]:
# Build full graph
G_full = nx.from_pandas_edgelist(edge_df, 'org_a', 'org_b', edge_attr='collaborations')
nx.set_node_attributes(G_full, org_attrs.set_index('organisationID').to_dict('index'))

print(f"Full graph — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")
print(f"Connected components: {nx.number_connected_components(G_full)}")

# Subgraph: top 60 orgs by degree
top_nodes = sorted(G_full.degree(), key=lambda x: x[1], reverse=True)[:60]
top_node_ids = [n for n, _ in top_nodes]
G = G_full.subgraph(top_node_ids).copy()
print(f"\nSubgraph (top 60) — nodes: {G.number_of_nodes()}  edges: {G.number_of_edges()}")

# Compute layout once — reused in all subsequent plots
pos = nx.spring_layout(G, seed=42, k=2.5, weight='collaborations')

# Labels for the top 20 nodes
top20_ids = [n for n, _ in top_nodes[:20]]
labels = {n: G.nodes[n].get('label', str(n)) for n in top20_ids}

# Basic draw
fig, ax = plt.subplots(figsize=(14, 10))
nx.draw(G, pos, ax=ax,
        node_color='steelblue', node_size=300,
        edge_color='lightgray', width=0.8,
        with_labels=False, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, font_color='black', ax=ax)
ax.set_title(f"CORDIS Collaboration Network — Top 60 Orgs\nMaster call: {MASTER_CALL}", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 5. Edge Width by Number of Collaborations

Thicker edges = more shared projects between the two organisations.

In [ ]:
collab_counts = nx.get_edge_attributes(G, 'collaborations')
max_collab = max(collab_counts.values()) if collab_counts else 1
edge_widths = [0.4 + 5.6 * (collab_counts[e] / max_collab) for e in G.edges()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(G, pos, node_color='steelblue', node_size=300, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='steelblue', alpha=0.5, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
ax.set_title("Edge Width ∝ Number of Shared Projects", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 6. Node Size by EC Funding

Larger nodes have received more EC funding (summed across all projects in this call).

In [ ]:
funding = {n: G.nodes[n].get('total_ec', 0) or 0 for n in G.nodes()}
max_funding = max(funding.values()) if max(funding.values()) > 0 else 1
node_sizes = [100 + 2400 * (funding[n] / max_funding) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='coral', alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', alpha=0.4, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
ax.set_title("Node Size ∝ Total EC Funding Received (€)", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 7. Node Colour by Country

Each colour represents the country of origin, derived from the 2-letter VAT number prefix.
Nodes with no country data are shown in grey.

In [ ]:
countries = [G.nodes[n].get('country') for n in G.nodes()]
unique_countries = sorted(set(c for c in countries if isinstance(c, str)))

cmap = plt.cm.get_cmap('tab20', max(len(unique_countries), 1))
country_color = {c: cmap(i) for i, c in enumerate(unique_countries)}
node_colors = [country_color.get(G.nodes[n].get('country'), (0.7, 0.7, 0.7, 1.0))
               for n in G.nodes()]

# Legend: top 12 countries in subgraph
country_counts = Counter(c for c in countries if isinstance(c, str))
top_countries = [c for c, _ in country_counts.most_common(12)]
legend_patches = [mpatches.Patch(color=country_color[c], label=c) for c in top_countries]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=400, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.4, width=0.8, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
ax.legend(handles=legend_patches, loc='upper left', title='Country', fontsize=8, ncol=2)
ax.set_title("Node Colour by Country of Origin", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 8. Fully Styled Network

Combining all three encodings:

| Visual encoding | Data |
|---|---|
| **Node size** | Total EC funding received |
| **Node colour** | Country of origin |
| **Edge width** | Number of shared projects |

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', alpha=0.45, ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                       linewidths=0.5, edgecolors='white', ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, font_weight='bold', ax=ax)

# Country legend
country_legend_patches = [mpatches.Patch(color=country_color[c], label=c) for c in top_countries]
country_legend = ax.legend(handles=country_legend_patches, loc='upper left',
                           title='Country', fontsize=8, ncol=2, framealpha=0.9)
ax.add_artist(country_legend)

# Encoding key
encoding_patches = [
    mpatches.Patch(color='lightgray', label='Node size  = EC funding'),
    mpatches.Patch(color='lightgray', label='Edge width = # collaborations'),
]
ax.legend(handles=encoding_patches, loc='lower left', fontsize=8, framealpha=0.9)

ax.set_title(
    f"CORDIS Collaboration Network — Top 60 Orgs\nMaster call: {MASTER_CALL}",
    fontsize=14, fontweight='bold'
)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 9. Community Detection

The **Greedy Modularity** algorithm finds clusters of organisations that collaborate more
within the cluster than with the rest of the network.
These communities often reflect research consortia, geographic clusters, or thematic groups.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))
print(f"Communities found: {len(communities)}")
for i, c in enumerate(sorted(communities, key=len, reverse=True)):
    sample = [G.nodes[n].get('label', str(n)) for n in list(c)[:5]]
    suffix = '...' if len(c) > 5 else ''
    print(f"  Community {i+1:2d} ({len(c):3d} orgs): {', '.join(sample)}{suffix}")

community_map = {n: i for i, c in enumerate(communities) for n in c}
palette = plt.cm.get_cmap('Set2', len(communities))
comm_colors = [palette(community_map[n]) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=comm_colors,
                       linewidths=0.5, edgecolors='white', ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, font_weight='bold', ax=ax)

legend_patches = [
    mpatches.Patch(color=palette(i), label=f"Community {i+1} ({len(c)} orgs)")
    for i, c in enumerate(sorted(communities, key=len, reverse=True))
]
ax.legend(handles=legend_patches, loc='upper left', title='Community', fontsize=8, framealpha=0.9)
ax.set_title("Community Detection — Greedy Modularity Algorithm", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 10. Centrality Analysis

| Measure | What it captures in this context |
|---|---|
| **Degree** | Number of unique organisations collaborated with |
| **Betweenness** | Acts as a "bridge" between different research groups |
| **Eigenvector** | Connected to other highly-connected organisations |
| **PageRank** | Recursive influence score |

In [ ]:
degree_c   = nx.degree_centrality(G)
between_c  = nx.betweenness_centrality(G, weight='collaborations')
eigen_c    = nx.eigenvector_centrality(G, max_iter=1000, weight='collaborations')
pagerank_c = nx.pagerank(G, weight='collaborations')

top15 = sorted(degree_c, key=degree_c.get, reverse=True)[:15]
df_cent = pd.DataFrame({
    'Organisation': [G.nodes[n].get('label', str(n)) for n in top15],
    'Country':      [G.nodes[n].get('country', '?')  for n in top15],
    'Degree':       [round(degree_c[n],   3) for n in top15],
    'Betweenness':  [round(between_c[n],  3) for n in top15],
    'Eigenvector':  [round(eigen_c[n],    3) for n in top15],
    'PageRank':     [round(pagerank_c[n], 3) for n in top15],
})
print("Top 15 organisations by degree centrality:")
print(df_cent.to_string(index=False))

# Visualise betweenness centrality
bc_vals = np.array([between_c[n] for n in G.nodes()])
bc_range = bc_vals.max() - bc_vals.min()
bc_norm = (bc_vals - bc_vals.min()) / (bc_range if bc_range > 0 else 1)
node_sizes_bc = 200 + 2500 * bc_norm

fig, ax = plt.subplots(figsize=(14, 10))
nodes_sc = nx.draw_networkx_nodes(G, pos, node_size=node_sizes_bc,
                                   node_color=bc_vals, cmap=cm.plasma, ax=ax)
nx.draw_networkx_edges(G, pos, width=0.5, edge_color='gray', alpha=0.4, ax=ax)
top_bc_ids = sorted(between_c, key=between_c.get, reverse=True)[:15]
bc_labels = {n: G.nodes[n].get('label', str(n)) for n in top_bc_ids}
nx.draw_networkx_labels(G, pos, labels=bc_labels, font_size=7, font_weight='bold', ax=ax)
plt.colorbar(nodes_sc, ax=ax, label='Betweenness Centrality', shrink=0.7)
ax.set_title(
    "Node Size & Colour ∝ Betweenness Centrality\n(Larger/brighter = key bridge organisations)",
    fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 11. Interactive Visualisation (PyVis)

**PyVis** wraps the [vis.js](https://visjs.org/) JavaScript library and accepts NetworkX graphs directly.
The output is a **standalone HTML file** that you can:

- Open in any browser — hover over nodes to see details, drag to rearrange, scroll to zoom
- Embed on a website with a simple `<iframe>` tag

```html
<iframe src="cordis_network_interactive.html" width="100%" height="800px"></iframe>
```

In [ ]:
try:
    from pyvis.network import Network
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyvis"])
    from pyvis.network import Network

from matplotlib.colors import to_hex

# --- Prepare node attributes for PyVis ---
degrees = dict(G.degree())
max_ec = max((G.nodes[n].get('total_ec', 0) or 0) for n in G.nodes())

for n in G.nodes():
    ec = G.nodes[n].get('total_ec', 0) or 0
    country = G.nodes[n].get('country', '?')
    lbl = G.nodes[n].get('label', str(n))
    deg = degrees[n]

    # Hover tooltip (HTML)
    G.nodes[n]['title'] = (
        f"<b>{lbl}</b><br>"
        f"Country: {country}<br>"
        f"EC Funding: €{ec:,.0f}<br>"
        f"Degree: {deg}<br>"
        f"Edges in subgraph: {deg}"
    )

    # Node size (scaled 10–60)
    G.nodes[n]['size'] = 10 + 50 * (ec / max_ec) if max_ec > 0 else 20

    # Node colour by country (reuse existing mapping)
    G.nodes[n]['color'] = to_hex(country_color.get(country, (0.7, 0.7, 0.7, 1.0)))

    # Only label top 20 nodes to keep graph clean
    G.nodes[n]['label'] = lbl if n in top20_ids else ''

# --- Prepare edge attributes ---
for u, v, d in G.edges(data=True):
    collabs = d.get('collaborations', 1)
    d['width'] = 0.5 + 4.5 * (collabs / max_collab)
    d['title'] = f"{collabs} shared project{'s' if collabs != 1 else ''}"
    d['color'] = '#cccccc'

# --- Build PyVis network ---
net = Network(height="750px", width="100%", bgcolor="#ffffff",
              font_color="black", notebook=True, cdn_resources="in_line")
net.from_nx(G)

net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -30000,
      "centralGravity": 0.3,
      "springLength": 150,
      "springConstant": 0.04
    },
    "stabilization": {"iterations": 200}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "zoomView": true
  }
}
""")

OUTPUT_FILE = "cordis_network_interactive.html"
net.save_graph(OUTPUT_FILE)
print(f"Interactive visualisation saved to: {OUTPUT_FILE}")
print(f"\nEmbed on a website with:")
print(f'  <iframe src="{OUTPUT_FILE}" width="100%" height="800px"></iframe>')
net.show(OUTPUT_FILE)